In [0]:
catalog = dbutils.widgets.get("catalog")
schema_prefix = dbutils.widgets.get("schema_prefix")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema_prefix}_silver")

In [0]:
import mlflow
import mlflow.data
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# Load 10 samples from the silver table
table_name = f"{catalog}.{schema_prefix}_silver.crm_silver"
df = spark.table("crm_silver").limit(10).toPandas()

# Encode 'stage' as a numeric feature
le = LabelEncoder()
df["stage_encoded"] = le.fit_transform(df["stage"])

# Create a fake binary label (1 for first 5 rows, 0 for last 5)
df["label"] = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

# Fit logistic regression
X = df[["stage_encoded"]]
y = df["label"]

with mlflow.start_run(run_name="crm_logistic_regression") as run:
    # Log the input dataset for lineage
    dataset = mlflow.data.from_spark(
        spark.table("crm_silver").limit(10),
        table_name=table_name,
        version="0"
    )
    mlflow.log_input(dataset, context="training")

    # Train model
    model = LogisticRegression()
    model.fit(X, y)

    # Log parameters and metrics
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("features", "stage_encoded")
    mlflow.log_metric("training_accuracy", model.score(X, y))

    # Log model with signature and input_example (required for UC registration)
    from mlflow.models import infer_signature
    signature = infer_signature(X, model.predict(X))

    model_name = f"{catalog}.{schema_prefix}_gold.crm_lead_classifier"
    mlflow.sklearn.log_model(
        model,
        name="model",
        signature=signature,
        input_example=X[:3],
        registered_model_name=model_name,
    )

    print(f"Run ID: {run.info.run_id}")
    print(f"Training accuracy: {model.score(X, y):.2f}")
    print(f"Model registered as: {model_name}")
